In [5]:
from langgraph.graph import MessagesState
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv, find_dotenv 
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver

In [6]:
load_dotenv(find_dotenv())

True

In [7]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [8]:
class ChatState(MessagesState):
    summary: str

In [9]:
def summarize_conversation(state: ChatState):

    existing_summary = state["summary"]

    # Build summarization prompt
    if existing_summary:
        prompt = (
            f"Existing summary:\n{existing_summary}\n\n"
            "Extend the summary using the new conversation above."
        )
    else:
        prompt = "Summarize the conversation above."

    messages_for_summary = state["messages"] + [
        HumanMessage(content=prompt)
    ]

    response = model.invoke(messages_for_summary)

    # Keep only last 2 messages verbatim
    messages_to_delete = state["messages"][:-2]

    return {
        "summary": response.content,
        "messages": [RemoveMessage(id=m.id) for m in messages_to_delete],
    }

In [10]:
def chat_node(state: ChatState):
    messages = []

    if state["summary"]:
        messages.append({
            "role": "system",
            "content": f"Conversation summary:\n{state['summary']}"
        })

    messages.extend(state["messages"])

    print(messages)

    response = model.invoke(messages)
    return {"messages": [response]}

In [11]:
def should_summarize(state: ChatState):
    return len(state["messages"]) > 6

In [12]:
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)
builder.add_node("summarize", summarize_conversation)

builder.add_edge(START, "chat")

builder.add_conditional_edges(
    "chat",
    should_summarize,
    {
        True: "summarize",
        False: "__end__",
    }
)

builder.add_edge("summarize", "__end__")

In [13]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [14]:
graph

In [15]:
config = {"configurable": {"thread_id": "t1"}}

def run_turn(text: str):
    out = graph.invoke({"messages": [HumanMessage(content=text)], "summary": ""}, config=config)
    return out

In [16]:
# gives the current version of the state
def show_state():
    snap = graph.get_state(config)
    vals = snap.values
    print("\n--- STATE ---")
    print("summary:", vals.get("summary", ""))
    print("num_messages:", len(vals.get("messages", [])))
    print("messages:")
    for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content[:80])

In [17]:
run_turn('Quantum Physics')
show_state()

[HumanMessage(content='Quantum Physics', id='8f559f2f-26aa-4636-90c4-49851d492af5')]

--- STATE ---
summary: 
num_messages: 2
messages:
- HumanMessage : Quantum Physics
- AIMessage : Quantum Physics is a branch of physics that deals with the behavior of matter an


In [18]:
run_turn('How is Albert Einstien related?')
show_state()

[HumanMessage(content='Quantum Physics', id='8f559f2f-26aa-4636-90c4-49851d492af5'), AIMessage(content='Quantum Physics is a branch of physics that deals with the behavior of matter and energy at the atomic and subatomic levels. It\'s one of the most successful and profound theories in human history, fundamentally changing our understanding of reality, even though its concepts often defy classical intuition.\n\nHere\'s a breakdown of its core aspects:\n\n1.  **The Scale:** Quantum physics applies to the very small – atoms, electrons, protons, neutrons, photons, and other elementary particles. At this scale, the rules that govern the macroscopic world (classical physics) break down.\n\n2.  **Quantization:** This is where the "quantum" in quantum physics comes from. Energy, momentum, angular momentum, and other physical properties are not continuous but come in discrete packets, or "quanta." For example, light isn\'t just any arbitrary wave; it\'s composed of individual energy packets ca

In [ ]:
run_turn('What are some of Einstien"s famous work')
show_state()

[HumanMessage(content='Quantum Physics', id='8f559f2f-26aa-4636-90c4-49851d492af5'), AIMessage(content='Quantum Physics is a branch of physics that deals with the behavior of matter and energy at the atomic and subatomic levels. It\'s one of the most successful and profound theories in human history, fundamentally changing our understanding of reality, even though its concepts often defy classical intuition.\n\nHere\'s a breakdown of its core aspects:\n\n1.  **The Scale:** Quantum physics applies to the very small – atoms, electrons, protons, neutrons, photons, and other elementary particles. At this scale, the rules that govern the macroscopic world (classical physics) break down.\n\n2.  **Quantization:** This is where the "quantum" in quantum physics comes from. Energy, momentum, angular momentum, and other physical properties are not continuous but come in discrete packets, or "quanta." For example, light isn\'t just any arbitrary wave; it\'s composed of individual energy packets ca

In [20]:
run_turn('Explain special theory of relativity')
show_state()

[HumanMessage(content='Quantum Physics', id='8f559f2f-26aa-4636-90c4-49851d492af5'), AIMessage(content='Quantum Physics is a branch of physics that deals with the behavior of matter and energy at the atomic and subatomic levels. It\'s one of the most successful and profound theories in human history, fundamentally changing our understanding of reality, even though its concepts often defy classical intuition.\n\nHere\'s a breakdown of its core aspects:\n\n1.  **The Scale:** Quantum physics applies to the very small – atoms, electrons, protons, neutrons, photons, and other elementary particles. At this scale, the rules that govern the macroscopic world (classical physics) break down.\n\n2.  **Quantization:** This is where the "quantum" in quantum physics comes from. Energy, momentum, angular momentum, and other physical properties are not continuous but come in discrete packets, or "quanta." For example, light isn\'t just any arbitrary wave; it\'s composed of individual energy packets ca

C:\Users\saroh\AppData\Local\Temp\ipykernel_1228\431833766.py:25: LangChainBetaWarning: The class `RemoveMessage` is in beta. It is actively being worked on, so the API may change.
  "messages": [RemoveMessage(id=m.id) for m in messages_to_delete],
